# Real-time Distillation

The challenge: full diffusion needs 20-50 denoising steps per frame. AR adds sequential dependency between frames. To get interactive framerates (~30 FPS), you need 1-4 steps per frame AND fast sequential generation.

This is fundamentally hard because:
- **Fewer steps = more approximation error per frame.** Each denoising step refines the image; cutting steps means less refinement.
- **AR compounds this error across frames.** Frame N's approximation error becomes part of frame N+1's input.
- **Distillation can transfer multi-step quality to fewer steps, but loses fine details.** The student model has less capacity to represent the full denoising trajectory.
- **Nobody has solved this at production quality.** Current approaches trade off quality, speed, and temporal consistency -- you can optimize any two, but not all three.

**Time:** ~2-3 hrs. GPU needed for LCM demos and consistency model training.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

In [ ]:
# --- Load LCM-LoRA on Stable Diffusion 1.5 ---
# LCM (Latent Consistency Models) distills the multi-step diffusion process into
# fewer steps by training the model to predict the final clean latent directly.
# LCM-LoRA is a lightweight adapter that adds this capability to any SD model.
# GPU: ~4 GB VRAM in float16

from diffusers import DiffusionPipeline, LCMScheduler, EulerDiscreteScheduler

pipe = DiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)

# Load LCM-LoRA adapter
pipe.load_lora_weights("latent-consistency/lcm-lora-sdv1-5")
pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)
print("Pipeline loaded with LCM-LoRA")

In [ ]:
# --- Speed/Quality Comparison ---
# Base SD 1.5 at 20 steps vs LCM at 4/2/1 steps.
# LCM uses low guidance scale (1.0-2.0) because high CFG creates artifacts with few steps.

prompt = "a beautiful landscape painting of mountains at sunset"
seed = 42

configs = [
    ("Base SD (20 steps)", 20, 7.5, False),
    ("LCM (4 steps)", 4, 1.5, True),
    ("LCM (2 steps)", 2, 1.5, True),
    ("LCM (1 step)", 1, 1.5, True),
]

images = []
timings = []

for name, steps, cfg, use_lcm in configs:
    # Switch between base and LCM scheduler
    if not use_lcm:
        pipe.unload_lora_weights()
        pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
    else:
        try:
            pipe.load_lora_weights("latent-consistency/lcm-lora-sdv1-5")
        except Exception:
            pass  # Already loaded
        pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)

    gen = torch.Generator(device=device).manual_seed(seed)

    # Warmup
    _ = pipe(prompt, num_inference_steps=steps, guidance_scale=cfg, generator=gen)

    # Timed run
    gen = torch.Generator(device=device).manual_seed(seed)
    torch.cuda.synchronize() if device.type == "cuda" else None
    start = time.perf_counter()
    result = pipe(prompt, num_inference_steps=steps, guidance_scale=cfg, generator=gen)
    torch.cuda.synchronize() if device.type == "cuda" else None
    elapsed = time.perf_counter() - start

    images.append(result.images[0])
    timings.append(elapsed)
    print(f"{name}: {elapsed:.3f}s")

# Plot results
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, ((name, steps, cfg, _), img, elapsed) in enumerate(zip(configs, images, timings)):
    axes[i].imshow(img)
    speedup = timings[0] / elapsed if elapsed > 0 else 0
    axes[i].set_title(f"{name}\n{elapsed:.2f}s ({speedup:.1f}x vs base)")
    axes[i].axis("off")

plt.suptitle("Distillation Speed/Quality Tradeoff", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nSpeedups: {', '.join(f'{timings[0]/t:.1f}x' for t in timings)}")
print(f"Note: LCM also uses guidance_scale=1.5 vs base's 7.5")
print(f"This means LCM skips the classifier-free guidance computation (2x savings on top)")

In [ ]:
# --- Consistency Model: Toy Implementation on MNIST ---
# Key idea: train model so f(x_t, t) = f(x_t', t') for any two points on the
# same ODE trajectory. This means the model maps ANY noisy version of an image
# directly to the clean output in ONE step.
#
# The "consistency" property: if you start from the same clean image and add
# different amounts of noise, the model should recover the same clean image
# regardless of which noise level you feed in.
#
# GPU: ~1-2 GB VRAM

class ConsistencyModel(nn.Module):
    """Simplified consistency model on MNIST.
    
    Parameterization: output = c_skip(t) * x + c_out(t) * F(x, t)
    At t=0: c_skip=1, c_out=0 -> identity (already clean)
    At t=1: c_skip~0, c_out~1 -> fully predicted
    This boundary condition enforces self-consistency at t=0.
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28 + 128, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, 28 * 28),
        )
        self.time_embed = nn.Sequential(
            nn.Linear(1, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
        )

    def c_skip(self, t: torch.Tensor) -> torch.Tensor:
        """Skip connection coefficient: 1 at t=0 (identity), ~0 at t=1."""
        return 1 / (1 + t ** 2)

    def c_out(self, t: torch.Tensor) -> torch.Tensor:
        """Output coefficient: 0 at t=0, ~1 at t=1."""
        return t / (1 + t ** 2).sqrt()

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        t_emb = self.time_embed(t.view(B, 1))
        x_flat = x.view(B, -1)
        h = self.net(torch.cat([x_flat, t_emb], dim=1))
        h = h.view(B, 1, 28, 28)
        skip = self.c_skip(t).view(B, 1, 1, 1)
        out = self.c_out(t).view(B, 1, 1, 1)
        return skip * x + out * h


# Visualize the skip/output coefficients
t_range = torch.linspace(0, 1, 100)
cm_temp = ConsistencyModel()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_range.numpy(), cm_temp.c_skip(t_range).numpy(), label="c_skip(t) -- identity weight")
ax.plot(t_range.numpy(), cm_temp.c_out(t_range).numpy(), label="c_out(t) -- network weight")
ax.set_xlabel("Noise level t")
ax.set_ylabel("Coefficient")
ax.set_title("Consistency Model Parameterization\nAt t=0: pure identity. At t=1: pure network prediction.")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Train with consistency distillation
mnist = datasets.MNIST(
    "./data", train=True, download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]),
)
loader = DataLoader(mnist, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

model_cm = ConsistencyModel().to(device)
# EMA target model (provides stable targets during training)
target_model = ConsistencyModel().to(device)
target_model.load_state_dict(model_cm.state_dict())
for p in target_model.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.Adam(model_cm.parameters(), lr=1e-4)

print(f"Consistency model params: {sum(p.numel() for p in model_cm.parameters()):,}")
print("Training consistency model...")

cm_losses = []
for epoch in tqdm(range(15), desc="CM Training"):
    epoch_losses = []
    for x, _ in loader:
        x = x.to(device)

        # Sample two adjacent timesteps on the ODE trajectory
        t = torch.rand(x.shape[0], device=device) * 0.98 + 0.01  # [0.01, 0.99]
        t_next = (t - 0.05).clamp(min=0.01)

        # Add noise at both levels (same noise, different amounts)
        noise = torch.randn_like(x)
        x_t = (1 - t.view(-1, 1, 1, 1)) * x + t.view(-1, 1, 1, 1) * noise
        x_t_next = (1 - t_next.view(-1, 1, 1, 1)) * x + t_next.view(-1, 1, 1, 1) * noise

        # Consistency loss: f(x_t, t) should equal f(x_{t_next}, t_next)
        # Both points are on the same trajectory, so they should map to the same clean output
        pred = model_cm(x_t, t)
        with torch.no_grad():
            target = target_model(x_t_next, t_next)

        loss = F.mse_loss(pred, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # EMA update for target model
        with torch.no_grad():
            for p, tp in zip(model_cm.parameters(), target_model.parameters()):
                tp.data.lerp_(p.data, 0.005)

        epoch_losses.append(loss.item())

    avg = np.mean(epoch_losses)
    cm_losses.append(avg)
    if epoch % 5 == 0:
        print(f"  Epoch {epoch}: loss={avg:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(cm_losses)
plt.xlabel("Epoch")
plt.ylabel("Consistency Loss")
plt.title("Consistency Model Training")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Done!")

In [ ]:
# --- Single-Step Generation with Consistency Model ---
# The whole point: noise -> clean in ONE forward pass.
# Compare with multi-step refinement to see the quality/speed tradeoff.

@torch.no_grad()
def sample_consistency_1step(model: nn.Module, n_samples: int) -> torch.Tensor:
    """One-step generation: noise -> clean in a single forward pass."""
    noise = torch.randn(n_samples, 1, 28, 28, device=device)
    t = torch.ones(n_samples, device=device) * 0.99
    return model(noise, t).clamp(-1, 1)


@torch.no_grad()
def sample_consistency_multistep(model: nn.Module, n_samples: int, n_steps: int = 4) -> torch.Tensor:
    """Multi-step generation: iteratively denoise with the consistency model.
    
    At each step: predict clean -> add noise at lower level -> predict clean again.
    More steps = better quality but slower.
    """
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    timesteps = torch.linspace(0.99, 0.01, n_steps + 1, device=device)

    for i in range(n_steps):
        t_curr = timesteps[i].expand(n_samples)
        # Predict clean image
        x_clean = model(x, t_curr)

        if i < n_steps - 1:
            # Add noise at the next (lower) level
            t_next = timesteps[i + 1]
            noise = torch.randn_like(x)
            x = (1 - t_next) * x_clean + t_next * noise
        else:
            x = x_clean

    return x.clamp(-1, 1)


# Generate samples with different step counts
step_configs = [1, 2, 4, 8]
fig, axes = plt.subplots(1, len(step_configs), figsize=(20, 5))

for i, n_steps in enumerate(step_configs):
    torch.manual_seed(42)
    start = time.perf_counter()
    if n_steps == 1:
        samples = sample_consistency_1step(model_cm, 64)
    else:
        samples = sample_consistency_multistep(model_cm, 64, n_steps=n_steps)
    elapsed = time.perf_counter() - start

    grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
    axes[i].imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    axes[i].set_title(f"{n_steps} step{'s' if n_steps > 1 else ''} ({elapsed*1000:.0f}ms)")
    axes[i].axis("off")

plt.suptitle("Consistency Model: Step Count vs Quality", fontsize=14)
plt.tight_layout()
plt.show()

print("Observation: even 1 step produces recognizable digits.")
print("More steps refine details but with diminishing returns.")
print("This is the consistency property at work -- the model already knows the answer at t=0.99.")

## DMD: Distribution Matching Distillation (arXiv 2311.18828)

The starting point for modern distillation. Key idea:
- Instead of matching **individual samples** (like consistency models), match the **output distribution**
- Uses two score functions: one from the teacher (real diffusion model) and one from a learned "fake" score (trained on student outputs)
- Minimizes approximate KL divergence between student and teacher output distributions
- Achieves single-step ImageNet generation at near-teacher quality

**The DMD loss has two terms:**
1. **Distribution matching:** $\nabla_\theta D_{KL}(p_{student} \| p_{teacher})$ -- push student distribution toward teacher's
2. **Regression loss:** $\|G_\theta(z) - \text{sg}[\hat{x}_0]\|^2$ -- anchor to prevent mode collapse

The gradient of the KL is estimated using the difference between teacher and fake scores:
$$\nabla_\theta \approx \mathbb{E}[( s_{\text{fake}}(x_t, t) - s_{\text{teacher}}(x_t, t) ) \cdot \nabla_\theta x_t]$$

**General consensus in the field: "a good start but not a complete solution."**

Why? DMD was designed for images. Extending to video AR is the open problem:
- Video has temporal structure -- distributional matching per-frame isn't enough. You need to match the *joint* distribution over frame sequences.
- AR generation means errors from frame N affect frame N+1's input distribution. The student's distribution *shifts* over time.
- The "distribution" of video is much harder to characterize than images. What does it mean for two videos to be from the same distribution? Pixel-level? Motion-level? Semantic-level?

## CausVid: Bidirectional to Causal Distillation (CVPR 2025, arXiv 2412.07772)

Extends DMD-style distillation to video with a critical twist: converting bidirectional attention to causal.

**Setup:**
- **Teacher:** Trained bidirectional video diffusion model (all frames see all frames). High quality but can't stream.
- **Student:** Same architecture but with causal attention mask (frame t only sees frames 0..t-1). Can stream.

**Key innovations:**
1. **Asymmetric distillation:** Teacher uses full attention + 50 steps. Student uses causal attention + 4 steps. Two axes of compression simultaneously.
2. **DMD-style distribution matching adapted for video:** The "fake score" is trained on the student's video outputs, capturing temporal artifacts.
3. **Progressive training:** Start with many student steps, gradually reduce. The student first learns temporal consistency, then learns to maintain it with fewer steps.

**Result:** 50-step bidirectional -> 4-step causal. 1.3s initial latency, then streaming at 9.4 FPS.

**Why this matters for real-time video generation:** It's the bridge from diffusion (quality) to AR (streaming). You get the quality of a model that was trained with full bidirectional attention, but deployed with causal attention for real-time use.

## Causal Forcing

ODE-based distillation from an AR teacher model, focusing on **causal consistency**.

Key idea: enforce causality not just in attention masks but in the ODE trajectories themselves.

In standard diffusion, the denoising ODE for frame t is:
$$\frac{dx_t}{d\sigma} = \frac{x_t - D_\theta(x_t, \sigma)}{\sigma}$$

Causal forcing adds the constraint that $D_\theta(x_t, \sigma)$ for frame $t$ must be independent of frames $t+1, t+2, ...$. This is stronger than just masking attention:
- **Attention masking:** The network can't *directly* see future frames, but information can leak through shared normalization layers, positional encodings, etc.
- **Causal forcing:** The entire ODE trajectory for frame $t$ is mathematically independent of future frames. No information leakage possible.

This is important because when you generate frame-by-frame in AR mode, future frames literally don't exist yet. The model must produce the same output regardless of whether future frames will be appended.

## Rolling Forcing

Joint denoising of multiple frames with **progressive noise levels**.

```
Standard diffusion (all frames at same noise level):
  t=50:  [noisy, noisy, noisy, noisy, noisy, noisy, noisy, noisy]
  t=25:  [semi,  semi,  semi,  semi,  semi,  semi,  semi,  semi ]
  t=0:   [clean, clean, clean, clean, clean, clean, clean, clean]

Rolling forcing (progressive noise):
  step 1: [clean, light, medium, heavy, pure_noise, -, -, -]
  step 2: [clean, clean, light,  medium, heavy, pure_noise, -, -]
  step 3: [clean, clean, clean,  light,  medium, heavy, pure_noise, -]
```

**Key properties:**
- Leftmost frames are already clean and provide context for noisier frames
- All frames denoised simultaneously (parallel, unlike AR)
- Only need to keep a sliding window of frames in memory
- Enables multi-minute videos on a single GPU
- Bridge between parallel denoising (diffusion) and sequential (AR)

This is conceptually similar to how a painter works: the beginning of the painting is detailed while the end is still a rough sketch. Each "step" extends the detailed region forward.

## rCM: Rectified Consistency Models (ICLR 2026)

Combines two powerful ideas:

1. **Rectified flows:** Train the ODE to have **straight trajectories** from noise to data. Standard diffusion ODEs are curved, which means more steps are needed to follow them accurately. Straight trajectories can be traversed in fewer steps with less error.

2. **Consistency models:** Map any point on the trajectory to the endpoint in one step.

**The insight:** Consistency distillation works better when the underlying ODE has straight trajectories. Curved trajectories mean adjacent points on the trajectory can map to very different endpoints, making the consistency property harder to learn.

**rCM training:**
1. First, train a rectified flow model (straighten trajectories via iterative distillation)
2. Then, apply consistency distillation on the straightened trajectories
3. Use forward-reverse divergence as the distillation objective (tighter than standard consistency loss)

**Result:** 2-4 step video generation with near-teacher quality. The straighter trajectories mean each step covers more "distance" toward the clean output.

## The Open Questions

These are genuinely unsolved problems in the field:

**1. Quality-speed Pareto frontier is still steep.**
Going from 20 steps to 4 steps loses noticeable quality. Going to 1 step is much worse. The curve isn't smooth -- there's a "quality cliff" around 2-4 steps where small reductions in step count cause disproportionate quality drops. Can we flatten this curve? Nobody knows.

**2. Error accumulation in AR + distillation.**
Distillation introduces per-frame approximation error (the student doesn't perfectly match the teacher). AR compounds this error across frames. The interaction between these two error sources is not well understood. Is the total error additive? Multiplicative? Does it depend on the content?

**3. Temporal consistency with fewer steps.**
Fewer denoising steps means less opportunity for frames to be "corrected" for temporal consistency. With 50 steps, the model has 50 chances to align motion, lighting, and identity across frames. With 4 steps, it has 4. How to maintain coherent motion with minimal steps?

**4. Training stability.**
Distillation objectives (DMD, consistency) can be unstable, especially for video models. The fake score network in DMD can diverge. Consistency targets can oscillate. Hyperparameter sensitivity is high -- small changes in learning rate, EMA rate, or noise schedule can cause training collapse.

**5. Evaluation.**
How do you even measure if a real-time model is "good enough"? FVD (Frechet Video Distance) is noisy and doesn't correlate well with human perception. Human eval is slow and expensive. CLIP-based metrics don't capture temporal quality. There's no ImageNet-FID equivalent for video generation quality.

In [ ]:
# --- Distilled Video Model Comparison (if available) ---
# This cell attempts to compare undistilled vs distilled video generation.
# The key experiment: measure latency, visual quality, and temporal consistency.

try:
    from diffusers import AnimateDiffPipeline, MotionAdapter, LCMScheduler

    # Load AnimateDiff with motion adapter
    adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2")
    pipe_video = AnimateDiffPipeline.from_pretrained(
        "stable-diffusion-v1-5/stable-diffusion-v1-5",
        motion_adapter=adapter,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to(device)

    prompt_video = "a cat walking in a garden"

    # Base: 20 steps
    gen = torch.Generator(device=device).manual_seed(42)
    start = time.perf_counter()
    result_base = pipe_video(
        prompt_video, num_inference_steps=20, guidance_scale=7.5,
        num_frames=16, generator=gen,
    )
    time_base = time.perf_counter() - start

    # LCM: 4 steps
    pipe_video.load_lora_weights("latent-consistency/lcm-lora-sdv1-5")
    pipe_video.scheduler = LCMScheduler.from_config(pipe_video.scheduler.config)
    gen = torch.Generator(device=device).manual_seed(42)
    start = time.perf_counter()
    result_lcm = pipe_video(
        prompt_video, num_inference_steps=4, guidance_scale=1.5,
        num_frames=16, generator=gen,
    )
    time_lcm = time.perf_counter() - start

    # Visualize frames side by side
    fig, axes = plt.subplots(2, 8, figsize=(20, 5))
    for i in range(8):
        idx = i * 2
        axes[0, i].imshow(result_base.frames[0][idx])
        axes[0, i].set_title(f"t={idx}", fontsize=8)
        axes[0, i].axis("off")
        axes[1, i].imshow(result_lcm.frames[0][idx])
        axes[1, i].set_title(f"t={idx}", fontsize=8)
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel(f"Base\n{time_base:.1f}s", fontsize=10)
    axes[1, 0].set_ylabel(f"LCM\n{time_lcm:.1f}s", fontsize=10)
    plt.suptitle(f"Video: Base (20 steps) vs LCM (4 steps) -- {time_base/time_lcm:.1f}x speedup")
    plt.tight_layout()
    plt.show()

except ImportError as e:
    print(f"AnimateDiff not available: {e}")
    print("\nThe key experiment would be:")
    print("  1. Generate 16-frame video with base model (20 steps, ~30s)")
    print("  2. Generate 16-frame video with LCM (4 steps, ~6s)")
    print("  3. Compare:")
    print("     - Latency: expect ~5x speedup")
    print("     - Visual quality: LCM slightly softer, fewer fine details")
    print("     - Temporal consistency: LCM may have more flickering")
    print("     - CLIP score: similar (semantic content preserved)")
    print("     - Optical flow smoothness: LCM slightly worse (jerky motion)")
except Exception as e:
    print(f"Error loading video pipeline: {e}")
    print("This is expected on limited VRAM -- AnimateDiff needs ~6-8GB")

## Checkpoint

**Scenario:** Product wants 5x faster video generation. Your options and their unsolved tradeoffs:

**1. Fewer steps (scheduler optimization):**
DPM-Solver, Euler with fewer steps. Free speedup but quality drops noticeably below ~8 steps. Tradeoff: predictable, well-understood quality loss. Easy to A/B test. Good first move.

**2. Consistency distillation:**
1-4 steps with near-teacher quality for images. For video: CausVid shows promise but temporal consistency suffers. Tradeoff: significant training complexity (need teacher model, fake score network, careful EMA scheduling) + potential temporal artifacts that are hard to detect automatically.

**3. Architecture optimization:**
Sparse temporal attention, flash attention, torch.compile, quantization. 2-3x speedup with no quality loss. Tradeoff: engineering effort, hardware-specific optimizations, may not compose well with other speedups.

**4. AR + distillation (a major bet in the field):**
Stream frames with few steps each. Tradeoff: this IS the frontier -- error accumulation + quality gap is THE unsolved problem. High risk, high reward.

**5. Guidance distillation:**
Remove classifier-free guidance (which doubles compute) by distilling the guided behavior into the model weights. Tradeoff: less controllable at inference time -- you lose the ability to adjust guidance scale per-prompt. 2x speedup for free if you can accept fixed guidance.

**My recommendation:** Start with (3) for free wins, then (1) for quick gains, then invest in (2) or (4) for the big wins -- knowing the tradeoffs are still being researched. Layer them: architecture optimization + fewer steps + guidance distillation can compound to 5-10x without touching distillation.

**Further reading:**
- DMD (arXiv 2311.18828)
- CausVid (arXiv 2412.07772)
- LCM (arXiv 2310.04378)
- Consistency Models (arXiv 2303.01469)
- Rectified Consistency Models (ICLR 2026)